In [1]:

import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. Load Customer Churn dataset
df = pd.read_csv(
    r"C:\Users\ITW\Downloads\telco_churn.csv"
)

# 2. Clean column names
df.columns = df.columns.str.strip()

# 3. Remove customer ID because it is not useful for prediction
df = df.drop(columns=["customerID"], errors="ignore")

# 4. Clean target column
df = df.dropna(subset=["Churn"]).copy()
df["Churn"] = df["Churn"].astype(str).str.strip()

# Convert Yes/No into 1/0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df = df.dropna(subset=["Churn"])
df["Churn"] = df["Churn"].astype(int)

# 5. Separate features and target
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Convert TotalCharges to numeric
if "TotalCharges" in X.columns:
    X["TotalCharges"] = pd.to_numeric(
        X["TotalCharges"], errors="coerce"
    )

# 6. Identify categorical and numerical columns
categorical = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical = X.select_dtypes(
    include=["number"]
).columns.tolist()

# 7. Preprocessing
preprocessor = ColumnTransformer([
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical
    ),
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]),
        numerical
    )
])

# 8. Create Decision Tree model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        random_state=42,
        class_weight="balanced"
    ))
])

# 9. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 10. Train model
model.fit(X_train, y_train)

# 11. Evaluate model
predictions = model.predict(X_test)

print("Customer Churn Decision Tree Results")
print("------------------------------------")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print("\nClassification Report:")
print(classification_report(
    y_test,
    predictions,
    target_names=["Stay", "Churn"],
    zero_division=0
))

# 12. Save model for Flask application
with open("churn_model.pkl", "wb") as file:
    pickle.dump({
        "model": model,
        "features": X.columns.tolist()
    }, file)

print("\nModel saved successfully as churn_model.pkl")
print("Target: 1 = Churn, 0 = Stay")


C:\Users\ITW\AppData\Local\Temp\ipykernel_11504\2213311855.py:43: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = X.select_dtypes(


Customer Churn Decision Tree Results
------------------------------------
Training rows: 5634
Testing rows: 1409
Accuracy: 0.7551

Classification Report:
              precision    recall  f1-score   support

        Stay       0.90      0.75      0.82      1035
       Churn       0.53      0.76      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.76      0.77      1409


Model saved successfully as churn_model.pkl
Target: 1 = Churn, 0 = Stay


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,...,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,...,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,...,0,7,9,1,1,1,1,1,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699996,59.166.0.8,12520,149.171.126.6,31010,tcp,FIN,0.020383,320,1874,31,...,0,8,20,7,5,1,1,4,NaN,0
699997,59.166.0.0,18895,149.171.126.9,80,tcp,FIN,1.402957,19410,1087890,31,...,0,1,1,2,7,2,2,2,NaN,0
699998,59.166.0.0,30103,149.171.126.5,5190,tcp,FIN,0.007108,2158,2464,31,...,0,13,13,6,7,2,1,2,NaN,0
699999,59.166.0.6,30388,149.171.126.5,111,udp,CON,0.004435,568,304,31,...,0,10,13,6,5,1,1,3,NaN,0
